In [ ]:
!pip install roboflow ultralytics sahi

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="mz3cNkxiO8av9JAjZbS3")
project = rf.workspace("my-ws-lwkgs").project("tl_detector-coivv")
version = project.version(27)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to tl_detector-27 in coco:: 100%|██████████| 1374/1374 [00:11<00:00, 120.96it/s]


In [3]:
from sahi.slicing import slice_coco
from sahi.utils.file import save_json
from sahi.utils.coco import Coco
import os

out_dir = f"{dataset.location}/sliced"
try:
    os.rmdir(out_dir)
except:
    pass

coco_dict, coco_path = slice_coco(
    coco_annotation_file_path=f"{dataset.location}/train/_annotations.coco.json",
    image_dir=f"{dataset.location}/train",
    output_coco_annotation_file_name="annotations",
    output_dir=f"{out_dir}/images",
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    min_area_ratio=0.1,  # Add this
    ignore_negative_samples=False,
)

coco = Coco.from_coco_dict_or_path(coco_dict, image_dir=f"{out_dir}/images")
result = coco.split_coco_as_train_val(train_split_rate=0.85)

Loading coco annotations: 100%|██████████| 25510/25510 [00:00<00:00, 32103.49it/s]


In [4]:
from sahi.utils.coco import Coco, export_coco_as_yolo

sliced_yolo_dir=f"{out_dir}/yolo_dataset"
try:
    os.rmdir(sliced_yolo_dir)
except:
    pass

data_yml_path = export_coco_as_yolo(
    output_dir=sliced_yolo_dir,
    train_coco=result["train_coco"],
    val_coco=result["val_coco"]
)

2025-10-13 08:59:53,684 - sahi - INFO - generating image symlinks and annotation files for yolo... (coco.py:1576)
100%|██████████| 21683/21683 [01:10<00:00, 308.84it/s]
2025-10-13 09:01:03,895 - sahi - INFO - generating image symlinks and annotation files for yolo... (coco.py:1576)
100%|██████████| 3827/3827 [00:12<00:00, 311.61it/s]


In [ ]:
from ultralytics import YOLO

#sliced_yolo_dir="/workspace/tl_detector-19/sliced/yolo_dataset"

model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{sliced_yolo_dir}/data.yml",
    epochs=100,
    imgsz=320,
    rect=False,
    multi_scale=False,
    batch=128,
    workers=4,
    name='tl_detector',
    augment=True,  # включаем ручной контроль над аугментацией
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    auto_augment='none',
    erasing=0.0
)

Ultralytics 8.3.213 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24124MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=none, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/tl_detector-27/sliced/yolo_dataset/data.yml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=tl_detector4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7185cd005080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7185cd005080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

     32/150      4.89G      1.079     0.7299     0.9537         16        320: 100% ━━━━━━━━━━━━ 170/170 2.7it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.0it/s 3.0s
                   all       3827       1112      0.885      0.777      0.895      0.637

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


In [ ]:
print(f"Result: {results.save_dir}/confusion_matrix_normalized.png")
print(f"Model: {results.save_dir}/weights/best.pt")
print(f"Calibration images: {out_dir}/images")

In [ ]:
import time
import functools
import requests

TOKEN = "8212098701:AAHKTZpdO8FRoxQF9bJBHte_EupMDWWD3Ls"
CHAT_ID = "390672240"

def notify(text):
    requests.get(f"https://api.telegram.org/bot{TOKEN}/sendMessage", params={
        "chat_id": CHAT_ID,
        "text": text
    })

notify('Finished')